In [3]:
import clingo
import os
import pandas as pd
import textwrap
import numpy

In [4]:
# Path to the folder containing the CSV files
folder_path = 'data'
dataframes = []
def readDatasInPanda(file_path):
#we probaply should not combine all the files in one dataset just now for testing
    df = pd.read_csv(file_path)

        # Append the cleaned DataFrame to the list
    dataframes.append(df)

# Combine all DataFrames into one
    pandas_data = pd.concat(dataframes, ignore_index=True)

# Display the first few rows of the combined DataFrame
    return pandas_data
    #voltages_over_time = combined_data['ex1.r_load.v']
# Save the combined DataFrame to a new CSV file (optional)
#combined_data.to_csv('combined_data.csv', index=False)

In [5]:
def classify_voltage(value: float) -> str:
    """
    Map a real-valued voltage to one of the qualitative labels:
      vlow < (vmid,vlow) < vmid < (vmax,vmid) < vmax
    """
    return int(value)
    if value >= 12.6:
        return "vmax"
    elif value >= 11.0:
        return "(vmax,vmid)"
    elif value >= 8:
        return "vmid"
    elif value >= 7:
        return "(vmid,vlow)"
    else:
        return "vlow"


In [6]:
DOMAIN_ASP = textwrap.dedent("""% Constants
% Constants
% ********************
% 1. Define Constants
% ********************

vmin(f).
vmax(950).

% ********************
% 2. Define Entities
% ********************

% Batteries
battery(b1). 
battery(b2). 
battery(b3).

% Switches
switch(switch1). 
switch(switch2). 
switch(switch3).

% ********************
% 3. Define Connections
% ********************

% Each switch is connected to a specific battery
connected(switch1, b1).
connected(switch2, b2).
connected(switch3, b3).

% ********************
% 4. Define Battery Properties
% ********************

% Battery voltages at each time step
battery_voltage(b1, 12, 0).
battery_voltage(b2, 11, 0).
battery_voltage(b3, 13, 0).

% Expected voltages for fault detection
expected_voltage(b1, 10).
expected_voltage(b2, 10).
expected_voltage(b3, 10).


% ********************
% %. Controller Logic
% ********************

% Controller: Activate switches when voltage < vmin at time T
activate_switch(S, T) :- 
    total_voltage(V, T), 
    vmin(Vmin),        
    V < Vmin,          
    switch(S),
    switch_state(S, off, T),
    can_activate(S),
    time(T).

% Controller: Deactivate switches when voltage > vmax at time T
deactivate_switch(S, T) :- 
    total_voltage(V, T),
    vmax(Vmax),        
    V > Vmax,          
    switch(S),
    switch_state(S, on, T),
    can_deactivate(S),
    time(T).

% Activation order: switch1, then switch2, then switch3
can_activate(switch1).
can_activate(switch2) :- switch_state(switch1, on, T), time(T).
can_activate(switch3) :- switch_state(switch2, on, T), time(T).

% Deactivation order: switch3, then switch2, then switch1
can_deactivate(switch3).
can_deactivate(switch2) :- switch_state(switch3, off, T), time(T).
can_deactivate(switch1) :- switch_state(switch2, off, T), time(T).

% Apply activation to the next time step
switch_state(S, on, T+1) :- activate_switch(S, T).

% Apply deactivation to the next time step
switch_state(S, off, T+1) :- deactivate_switch(S, T).

% Carry forward the state if no change
switch_state(S, State, T+1) :- 
    switch_state(S, State, T),
    not activate_switch(S, T),
    not deactivate_switch(S, T),
    time(T+1).

% ********************
% &. Fault Detection
% ********************

% Battery Faults at each time step
battery_fault(S, T) :- 
    connected(S, B),
    switch_state(S, on, T),
    battery_voltage(B, Vb, T),
    Vb < expected_voltage(B),
    time(T).

% Switch Faults at each time step
switch_fault(S, T) :- 
    total_voltage(V_0, T),
    total_voltage(V_1, T + 1),
    activate_switch(S, T),
    switch_state(S, off, T),
    V_1 <= V_0,
    time(T).

switch_fault(S, T) :- 
    deactivate_switch(S, T),
    switch_state(S, on, T+1),
    time(T).
""")

In [18]:
def build_facts_from_csv(pandasData) -> str:
    """
    Reads time, Vbattery1, Vbattery2, Vbattery3, Vout, switch1, switch2, switch3 from a CSV file,
    maps Vout to qualitative states, and returns ASP facts as a string.
    """
    data = []
   

    for index, row in pandasData.iterrows():
        lines = []
        # Extract and classify output voltage
        t = index  # Convert time to integer
        vout_value = "Nan"
        vout_value = float(row["circ1.r_load.v"])*10  # Extract voltage value
        
        # Classify Vout into qualitative state
        vout_state = classify_voltage(vout_value)

        # Append time and voltage facts
        lines.append(f"time({t}).")
        lines.append(f"total_voltage({vout_state},{t}).")
        # Extract and process switch states
        switch1_state = row["circ1.s[1].mode"]
        switch2_state = row["circ1.s[2].mode"]
        switch3_state = row["circ1.s[3].mode"]  # Corrected key

        
        # Ensure switch states are either 'on' or 'off'
        #print( str(switch1_state) == "1.0")
        #print(switch1_state)
        switch1 = "on" if str(switch1_state) == "1.0" else "off"
        switch2 = "on" if str(switch2_state) == "1.0" else "off"
        switch3 = "on" if str(switch3_state) == "1.0" else "off"

        # Append switchState facts
        lines.append(f"switch_state(switch1,{switch1},{t}).")
        lines.append(f"switch_state(switch2,{switch2},{t}).")
        lines.append(f"switch_state(switch3,{switch3},{t}).")
        data.append(lines)
        #if index == 100: 
          #  break
    print(data)
    return data

In [19]:
def on_model(model, output):
    """
    Callback function to process each model found by Clingo.
    Collects the voltage assignments and any detected faults.
    """
    atoms = model.symbols(shown=True)
    battery_assignments = [str(a) for a in atoms if a.name == "voltage"]
   # heater_assignments = [str(a) for a in atoms if a.name == "heaterVoltage"]
    faults = [str(a) for a in atoms if a.name == "faulty"]

    output.append((battery_assignments, faults))

In [20]:
def live_diagnose_simulation(alls_data_facts, time_range):
    counter_max = len(alls_data_facts) - 1
    counter = 1
    
    while counter <= counter_max:
        # 1) Build ASP facts from the CSV
        # 2) Combine domain + data
        # Define time steps
        data_facts = []
        data_facts.append(f"time({counter-1}..{counter}).")
        data_facts.append("\n".join(alls_data_facts[counter - 1]))
        data_facts.append("\n".join(alls_data_facts[counter]))
        # todo split in time stamps 
        asp_data = "\n".join(data_facts)
        
        program = f"{DOMAIN_ASP}\n\n% Observed Data:\n{asp_data}\n"
    
        # 4) Optionally, print the complete program for debugging
        write_to_file = False
        if write_to_file:
            print("Complete ASP Program:\n")
            try:
                with open("diagnosis_program.asp", "w") as asp_file:
                    asp_file.write(program)
                print(f"ASP program written to '{"diagnosis_program.asp"}'.")
            except IOError as e:
                print(f"Error writing ASP program to file: {e}")
                return
       
        # 3) Initialize Clingo Control
        ctl = clingo.Control()
        
        # 4) Add the program
        ctl.add("base", [], program)
        ctl.ground([("base", [])])
       #+ print("Finished Init clingo\n")
        # 5) Prepare to collect solutions
        solutions = []
    
        # 6) Define the callback
        def callback(model):
            battery_assignments = []
            faults = []
            for atom in model.symbols(shown=True):
                if atom.name == "total_voltage":
                    battery_assignments.append(str(atom))
                elif atom.name == "battery_fault" or atom.name == "switch_fault":
                    faults.append(str(atom))
            solutions.append((battery_assignments, faults))
    
        # 7) Solve
        print("Start solving with clingo\n")
        ctl.solve(on_model=callback)
        #print("Finished solving with clingo\n")
        # 8) Process and Print Results
        if not solutions:
            print("No solutions found. Possible multiple faults or inconsistent data.")
        else:
          #  print(f"Found {len(solutions)} solution(s):\n")
            for idx, (bats, faults) in enumerate(solutions, start=1):
          #      print(f"--- Solution #{idx} ---")
         #       print("Battery Voltage Assignments:")
                #for bat in sorted(bats):
                   # print(f"  {bat}")
         #       print("Fault Hypotheses:")
                if faults:
                    for fault in faults:
                        print(f"  {fault}")
           #     else:
            #        print("  No faults detected.")
               # print() 
        counter += 1
    
        

In [21]:
def diagnose(csv_path: str, asp_output_path: str = "diagnosis_program.asp"):
    print("Finished reading data\n") 
    data_facts = build_facts_from_csv(readDatasInPanda(csv_path))
    live_diagnose_simulation(data_facts, 0)
      

In [22]:
diagnose("./data/Circuit_Scenario1_res.csv")

Finished reading data

[['time(0).', 'total_voltage(833,0).', 'switch_state(switch1,off,0).', 'switch_state(switch2,on,0).', 'switch_state(switch3,on,0).'], ['time(1).', 'total_voltage(833,1).', 'switch_state(switch1,off,1).', 'switch_state(switch2,on,1).', 'switch_state(switch3,on,1).'], ['time(2).', 'total_voltage(909,2).', 'switch_state(switch1,off,2).', 'switch_state(switch2,off,2).', 'switch_state(switch3,on,2).'], ['time(3).', 'total_voltage(909,3).', 'switch_state(switch1,off,3).', 'switch_state(switch2,off,3).', 'switch_state(switch3,on,3).'], ['time(4).', 'total_voltage(909,4).', 'switch_state(switch1,off,4).', 'switch_state(switch2,off,4).', 'switch_state(switch3,on,4).'], ['time(5).', 'total_voltage(909,5).', 'switch_state(switch1,off,5).', 'switch_state(switch2,off,5).', 'switch_state(switch3,on,5).'], ['time(6).', 'total_voltage(909,6).', 'switch_state(switch1,off,6).', 'switch_state(switch2,off,6).', 'switch_state(switch3,on,6).'], ['time(7).', 'total_voltage(909,7).', 's

In [23]:
readDatasInPanda("./data/Circuit_Scenario2_res.csv")["circ1.r_load.v"]

0      83.333336
1      83.333336
2      90.909091
3      90.909091
4      90.909091
         ...    
419    92.307694
420    92.307694
421    92.307694
422    92.307694
423    92.307694
Name: circ1.r_load.v, Length: 424, dtype: float64